# WAF Model Training - Improved Version

This notebook trains a model to classify web requests as normal or attack.
Addresses the precision issue where all URLs were being classified as attacks.

In [ ]:
# WAF Model Training - Improved Version
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the dataset
csv_file = "csic_database.csv"
df = pd.read_csv(csv_file)

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['classification'].value_counts())
print(f"\nClass distribution (%):")
print(df['classification'].value_counts(normalize=True) * 100)

In [ ]:
# Combine all text features into a single feature vector per request
# This approach is better than treating each field separately

def combine_features(row):
    """Combine all text features into a single string"""
    features = []
    
    # Add URL
    if pd.notna(row.get('URL')):
        features.append(str(row['URL']).strip())
    
    # Add Content
    if pd.notna(row.get('content')):
        features.append(str(row['content']).strip())
    
    # Add Cookie
    if pd.notna(row.get('cookie')):
        features.append(str(row['cookie']).strip())
    
    # Add User-Agent
    if pd.notna(row.get('User-Agent')):
        features.append(str(row['User-Agent']).strip())
    
    # Add Method
    if pd.notna(row.get('Method')):
        features.append(str(row['Method']).strip())
    
    # Combine all features with a separator
    combined = ' '.join(features)
    return combined.lower() if combined else ''

# Create combined feature
df['combined_text'] = df.apply(combine_features, axis=1)

# Remove rows with empty combined text
df = df[df['combined_text'].str.len() > 0].copy()

print(f"Dataset after feature combination: {df.shape}")
print(f"\nClass distribution after cleaning:")
print(df['classification'].value_counts())

In [ ]:
# Balance the dataset to ensure equal representation
from sklearn.utils import resample

df_normal = df[df['classification'] == 0].copy()
df_attack = df[df['classification'] == 1].copy()

print(f"Normal samples: {len(df_normal)}")
print(f"Attack samples: {len(df_attack)}")

# Balance by downsampling the majority class
n_samples = min(len(df_normal), len(df_attack))
print(f"\nBalancing to {n_samples} samples per class...")

df_normal_balanced = resample(
    df_normal,
    replace=False,
    n_samples=n_samples,
    random_state=42
)

df_attack_balanced = resample(
    df_attack,
    replace=False,
    n_samples=n_samples,
    random_state=42
)

df_balanced = pd.concat([df_normal_balanced, df_attack_balanced])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset shape: {df_balanced.shape}")
print(f"\nBalanced class distribution:")
print(df_balanced['classification'].value_counts())

In [ ]:
# Prepare features and labels
X = df_balanced['combined_text'].values
y = df_balanced['classification'].values

# Split into train and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining class distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nTest class distribution:")
print(pd.Series(y_test).value_counts())

In [ ]:
# Create a better model with class weights to handle any remaining imbalance
# Using character-level n-grams which work well for URL/HTTP request classification

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char',           # Character-level analysis works better for URLs
        ngram_range=(1, 3),         # 1-3 character n-grams
        max_features=2000,         # Increased features for better discrimination
        min_df=2,                  # Ignore very rare features
        max_df=0.95,               # Ignore very common features
        sublinear_tf=True          # Apply sublinear tf scaling
    )),
    ('classifier', SVC(
        kernel='rbf',
        C=1.0,                     # Regularization parameter
        gamma='scale',
        class_weight='balanced',   # CRITICAL: Automatically balance class weights
        probability=True,          # Enable probability estimates
        random_state=42
    ))
])

print("Model pipeline created with balanced class weights")

In [ ]:
# Train the model
print("Training model...")
model.fit(X_train, y_train)
print("Model training completed!")

In [ ]:
# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nTrue Negatives (Normal correctly classified): {cm[0][0]}")
print(f"False Positives (Normal misclassified as Attack): {cm[0][1]}")
print(f"False Negatives (Attack misclassified as Normal): {cm[1][0]}")
print(f"True Positives (Attack correctly classified): {cm[1][1]}")

print("\n" + "=" * 60)
print("METRICS")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")

In [ ]:
# Analyze prediction probabilities to understand model confidence
print("=" * 60)
print("PREDICTION PROBABILITY ANALYSIS")
print("=" * 60)

# Get probabilities for the attack class (class 1)
attack_probs = y_pred_proba[:, 1]

print(f"\nAttack probability statistics:")
print(f"Mean: {attack_probs.mean():.4f}")
print(f"Median: {np.median(attack_probs):.4f}")
print(f"Min: {attack_probs.min():.4f}")
print(f"Max: {attack_probs.max():.4f}")
print(f"Std: {attack_probs.std():.4f}")

# Check distribution of predictions
print(f"\nPrediction distribution:")
print(f"Predicted as Normal (0): {np.sum(y_pred == 0)}")
print(f"Predicted as Attack (1): {np.sum(y_pred == 1)}")

# Show some examples
print(f"\nSample predictions (first 10):")
for i in range(min(10, len(X_test))):
    pred_label = "Attack" if y_pred[i] == 1 else "Normal"
    true_label = "Attack" if y_test[i] == 1 else "Normal"
    prob = attack_probs[i]
    status = "✓" if y_pred[i] == y_test[i] else "✗"
    print(f"{status} True: {true_label:6s} | Pred: {pred_label:6s} | Prob: {prob:.3f}")

In [ ]:
# Cross-validation to ensure model stability
print("=" * 60)
print("CROSS-VALIDATION")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')

print(f"\nF1-Score across 5 folds:")
for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean F1-Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

In [ ]:
# Save the trained model
model_filename = "waf_improved_model.pkl"
joblib.dump(model, model_filename)
print(f"Model saved to {model_filename}")

# Also save a function to make predictions
def predict_request(text):
    """Predict if a request is an attack or normal"""
    if isinstance(text, str) and len(text.strip()) > 0:
        prediction = model.predict([text.lower()])[0]
        probability = model.predict_proba([text.lower()])[0]
        return {
            'prediction': 'attack' if prediction == 1 else 'normal',
            'confidence': float(max(probability)),
            'attack_probability': float(probability[1])
        }
    else:
        return {'prediction': 'normal', 'confidence': 0.0, 'attack_probability': 0.0}

# Test the prediction function
print("\nTesting prediction function:")
test_urls = [
    "http://localhost:8080/tienda1/index.jsp",
    "http://localhost:8080/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja",
    "http://localhost:8080/tienda1/publico/anadir.jsp?id=3' OR '1'='1",
    "http://localhost:8080/tienda1/publico/anadir.jsp?id=3 UNION SELECT * FROM users"
]

for url in test_urls:
    result = predict_request(url)
    print(f"URL: {url[:60]}...")
    print(f"  Prediction: {result['prediction']}, Confidence: {result['confidence']:.3f}, Attack Prob: {result['attack_probability']:.3f}")
    print()

## Model Improvements Summary

This improved model addresses the precision issue by:

1. **Class Weight Balancing**: Using `class_weight='balanced'` in SVC to automatically adjust for class imbalance
2. **Better Feature Engineering**: Combining all text features into a single vector per request instead of treating them separately
3. **Improved Feature Extraction**: 
   - Character-level n-grams (1-3) which work better for URL/HTTP patterns
   - Increased max_features to 2000 for better discrimination
   - Added min_df and max_df to filter out noise
   - Sublinear TF scaling for better feature weighting
4. **Proper Data Balancing**: Ensuring equal representation of both classes in training data
5. **Comprehensive Evaluation**: Including confusion matrix, precision, recall, F1-score, and probability analysis

The model should now have much better precision and not classify everything as an attack.